# Tutorial 9 — Pretraining: Mixed Precision, Schedules & Scaling Laws

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part III — Pretraining**  
**Follows:** Tutorial 8 (Optimizers: SGD, Adam, AdamW & Batch Size)  
**Precedes:** Tutorial 10 (Distributed Training)

---

## What This Tutorial Covers

This tutorial takes everything built so far and assembles a complete
pretraining run. The three topics that make the difference between
a training loop that works and one that is compute-optimal:

- **Mixed precision training** — BF16 vs FP16, `torch.autocast`,
  `GradScaler`, and why you should almost always use BF16 for modern
  hardware. The numerical argument, not just the recommendation.
- **Learning rate schedules** — linear warmup + cosine decay, derived
  from first principles. The intuition behind warmup. Implementing
  `CosineWithWarmup` from scratch. The `min_lr` choice.
- **Scaling laws** — the Chinchilla result. The $L(N, D)$ loss surface.
  The *compute-optimal* frontier $D \approx 20N$. Using the law to plan
  a training run given a fixed FLOP budget. What it means for the nano
  model.

At the end: a complete, instrumented pretraining loop that plugs in
the data pipeline from Tutorial 7, the monitoring from Tutorials 4–6,
and the model from Tutorial 2.

---

## 1. Mixed Precision Training

### The floating point formats

A standard float32 number uses 32 bits: 1 sign, 8 exponent, 23 mantissa.
The mantissa gives you 7 decimal digits of precision.

| Format | Bits | Exponent | Mantissa | Max value | Precision |
|---|---|---|---|---|---|
| FP32   | 32  | 8        | 23       | 3.4e38    | ~7 decimal digits |
| FP16   | 16  | 5        | 10       | 65504     | ~3 decimal digits |
| BF16   | 16  | 8        | 7        | 3.4e38    | ~2 decimal digits |

**The key difference between FP16 and BF16:**

FP16 has a maximum representable value of 65504. Activations and gradients
in a deep network can easily exceed this — especially early in training
when the loss is large. When a value overflows FP16, it becomes `inf`,
which propagates to `nan`, which kills the run. This is why FP16 requires
`GradScaler` — it multiplies the loss by a large constant before backward
to keep gradients in the representable range.

[BF16 has the same 8-bit exponent as FP32, giving it the same dynamic range
($3.4 \times 10^{38}$). It sacrifices mantissa precision (2 decimal digits
instead of 7) but can never overflow.]{.mark} For neural network training, where
values can span many orders of magnitude but precision below 1% rarely
matters, BF16 is strictly better than FP16 on hardware that supports it
(A100, H100, recent consumer cards).

In [ ]:
import torch

# Demonstrate the overflow problem with FP16
x = torch.tensor(70000.0)
print(x.to(torch.float16))   # tensor(inf) — overflow!
print(x.to(torch.bfloat16))  # tensor(69632.) — representable, slightly rounded

# Gradient underflow in FP16
small_grad = torch.tensor(1e-8)
print(small_grad.to(torch.float16))   # tensor(0.) — underflow to zero!
print(small_grad.to(torch.bfloat16))  # tensor(1.0000e-08) — preserved

### `torch.autocast`

`torch.autocast` is a context manager that casts operations to a lower
precision automatically. It is selective — operations that are sensitive
to precision (e.g., softmax, layer norm) stay in FP32; operations that
benefit from low-precision speed (matmul, conv) are cast to BF16/FP16:

In [ ]:
import torch
import torch.nn as nn

model = GPT(config).to(device)

# BF16 mixed precision — recommended for A100/H100 and recent consumer GPUs
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    logits, loss = model(x, y)
    # Inside the context: matmul ops run in BF16
    # Outside the context (or for sensitive ops): FP32 is used

# The loss is in FP32 (autocast doesn't cast the loss)
loss.backward()
optimizer.step()

**What autocast does NOT do:**
- It does not change the dtype of model parameters — they stay FP32
- It does not change the optimizer state — Adam moments stay FP32
- It does not affect CPU operations

The model parameters remain FP32 throughout training. `autocast` only
affects the intermediate computation within the forward pass. The gradient
that flows back through `.backward()` is also FP32 (the autocast context
only covers the forward pass).

### `GradScaler` for FP16

If you use FP16 instead of BF16, you need `GradScaler` to prevent gradient
underflow. The scaler multiplies the loss by a large scalar $S$ before
backward — this scales all gradients up by $S$, keeping them representable
in FP16. Before the optimizer step, it divides the gradients back by $S$:

In [ ]:
from torch.cuda.amp import GradScaler

scaler = GradScaler()   # starts with scale=65536 by default

for x, y in dataloader:
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        logits, loss = model(x, y)

    # Scale loss before backward
    scaler.scale(loss).backward()

    # Unscale gradients before clipping
    # (clip_grad_norm_ must see the true gradient magnitudes)
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # Step (automatically skips if gradients contain inf/nan)
    scaler.step(optimizer)
    scaler.update()   # adjusts scale factor based on whether step was skipped

The scaler doubles $S$ every 2000 steps where no inf/nan occurred, and
halves it whenever a step is skipped. This adaptive scaling finds the
largest $S$ that keeps gradients in range.

[**For BF16, `GradScaler` is not needed**]{.underline} — BF16's dynamic range matches
FP32, so gradients never underflow. Use:

In [ ]:
# BF16 — no GradScaler needed
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    logits, loss = model(x, y)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()

### CPU fallback

On CPU (no CUDA), `autocast` with BF16 is supported on recent PyTorch
versions but provides no speed benefit — CPUs don't have BF16 tensor cores.
Use `dtype=torch.float32` on CPU or skip autocast entirely:

In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
autocast_ctx = torch.autocast(device_type='cuda', dtype=dtype) \
               if torch.cuda.is_available() \
               else torch.no_grad().__class__()  # no-op context

---

## 2. Learning Rate Schedules

### Why warmup?

The Adam optimizer maintains a running estimate of the gradient mean ($m_t$)
and variance ($v_t$) for each parameter:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

At step 1, $m_0 = v_0 = 0$. The bias-corrected estimates are:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

[At $t=1$ with $\beta_1=0.9$, $\hat{m}_1 = m_1 / 0.1 = 10 m_1$ — the
first gradient is amplified 10× by the bias correction.]{.mark} The effective
step size is large and unstable until $m_t$ and $v_t$ have accumulated
enough history to be reliable estimates.

Warmup counteracts this: by starting with a very small LR (often 0) and
linearly increasing it over the first $T_{\text{warm}}$ steps, you give
the optimizer time to build up reliable momentum estimates before taking
large steps. In the first few steps, even with bias correction, the
adaptive step size is unreliable — warmup ensures the actual updates
remain small during this period.

**How many warmup steps?** A rule of thumb: 1–2% of total training steps.
For a 5000-step run: 50–100 warmup steps. For a 100K-step run: 1000–2000.
Longer warmup is safer but delays the start of real learning.

### Why cosine decay?

After warmup, the LR should decrease as training progresses. The intuition:
early in training, the model is far from a minimum and large steps help
navigate the loss landscape quickly. Late in training, the model is near
a minimum and large steps risk overshooting. A decreasing LR corresponds
to taking smaller, more precise steps as you approach a minimum.

Linear decay works but leaves the model at a non-zero LR at the end, wasting
the final steps. Exponential decay decays too fast — the LR becomes tiny
long before training ends. [Cosine decay has the right shape: slow decrease
early (when you can still move fast), rapid decrease in the middle (main
convergence phase), and a long flat tail approaching `min_lr`]{.underline} (fine-tuning
phase).

The cosine schedule with warmup:

$$\text{lr}(t) = \begin{cases}
\text{max\_lr} \cdot \frac{t}{T_{\text{warm}}} & t < T_{\text{warm}} \\
\text{min\_lr} + \frac{1}{2}(\text{max\_lr} - \text{min\_lr})\left(1 + \cos\frac{\pi(t - T_{\text{warm}})}{T_{\text{max}} - T_{\text{warm}}}\right) & t \geq T_{\text{warm}}
\end{cases}$$

In [ ]:
import math
from torch.optim.lr_scheduler import LambdaLR

def make_cosine_schedule(
    optimizer,
    max_lr:       float,
    min_lr:       float,
    warmup_steps: int,
    max_steps:    int,
) -> LambdaLR:
    """
    Linear warmup from 0 to max_lr over warmup_steps.
    Cosine decay from max_lr to min_lr over max_steps.

    The optimizer's initial lr is used as max_lr — set it to max_lr
    when constructing the optimizer.
    """
    def lr_lambda(step: int) -> float:
        # Return the multiplier on the optimizer's base lr
        if step < warmup_steps:
            # Linear warmup: 0 → 1.0
            return step / max(warmup_steps, 1)
        # Cosine decay: 1.0 → min_lr/max_lr
        progress = (step - warmup_steps) / max(max_steps - warmup_steps, 1)
        progress = min(progress, 1.0)   # clamp at 1.0 after max_steps
        cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
        # Scale from min_ratio to 1.0
        min_ratio = min_lr / max_lr
        return min_ratio + (1.0 - min_ratio) * cosine

    return LambdaLR(optimizer, lr_lambda)


# Visualize the schedule
def plot_lr_schedule(max_lr, min_lr, warmup_steps, max_steps):
    import matplotlib.pyplot as plt

    # Dummy optimizer to drive the scheduler
    dummy_param = torch.nn.Parameter(torch.zeros(1))
    opt   = torch.optim.AdamW([dummy_param], lr=max_lr)
    sched = make_cosine_schedule(opt, max_lr, min_lr, warmup_steps, max_steps)

    lrs = []
    for _ in range(max_steps):
        lrs.append(opt.param_groups[0]['lr'])
        sched.step()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(lrs, color='#9C27B0', lw=2)
    ax.axvline(warmup_steps, color='gray', linestyle='--', lw=0.8,
               label=f'warmup ends (step {warmup_steps})')
    ax.axhline(min_lr, color='gray', linestyle=':', lw=0.8,
               label=f'min_lr={min_lr:.1e}')
    ax.set_xlabel('Step')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Cosine Schedule with Linear Warmup')
    ax.legend()
    plt.tight_layout()
    plt.savefig('lr_schedule.png', dpi=150)
    plt.show()

plot_lr_schedule(
    max_lr=3e-4, min_lr=3e-5,
    warmup_steps=100, max_steps=5000
)

### Choosing `min_lr`

The Chinchilla paper and GPT practice converge on `min_lr = 0.1 × max_lr`.
The intuition: you want the optimizer to keep making meaningful updates
until the very end of training. If `min_lr` is too small (say, `1e-8`),
the last 20% of training steps barely move the weights — wasted compute.
If `min_lr` is too large (say, `0.5 × max_lr`), the schedule decays
insufficiently and the model does not converge as tightly.

In [ ]:
# Standard choices for the nano model
MAX_LR    = 3e-4    # peak learning rate
MIN_LR    = 3e-5    # 10% of max — standard Chinchilla convention
WARMUP    = 100     # 2% of 5000 steps
MAX_STEPS = 5000

---

## 3. Scaling Laws

### The Chinchilla result

Hoffmann et al. (2022) — the "Chinchilla" paper — trained hundreds of
language models of different sizes on different amounts of data and fit
a parametric model to the loss surface:

$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

where:
- $N$ = number of model parameters
- $D$ = number of training tokens
- $E \approx 1.69$ = irreducible entropy of natural language
- $A, B, \alpha, \beta$ = fitted constants

From this surface, given a compute budget $C \approx 6ND$ FLOPs
(6 FLOPs per parameter per token for a forward + backward pass),
the compute-optimal allocation is:

$$N^* \propto C^{0.5}, \quad D^* \propto C^{0.5}$$

which gives the famous rule of thumb:

$$D^* \approx 20 \cdot N$$

**Compute-optimal training means 20 tokens per parameter.**[^chinchilla]

[^chinchilla]: The Chinchilla result (Hoffmann et al., 2022) shows that for a fixed compute budget  pprox 6ND$ FLOPs, loss is minimized at ^* pprox 20N$ — train on 20 tokens per parameter. GPT-3 (175B params, 300B tokens) violated this: it was significantly undertrained. Llama and its descendants use Chinchilla-optimal ratios. A 10M
parameter model should be trained on ~200M tokens. GPT-3 (175B params)
was undertrained by this criterion — it saw 300B tokens, far fewer than
the 3.5T that Chinchilla predicts as optimal.

### Using the scaling laws

Given a fixed compute budget $C$ (measured in FLOPs), find the
compute-optimal model size and token count:

In [ ]:
def chinchilla_optimal(compute_flops: float) -> dict:
    """
    Given a compute budget in FLOPs, return the compute-optimal
    model size and token count.

    Uses Chinchilla fitted constants:
    A=406.4, B=410.7, alpha=0.34, beta=0.28
    (from Hoffmann et al. 2022, Table A3)
    """
    # C ≈ 6 * N * D  (6 FLOPs per param per token, fwd + bwd)
    # Optimal: N* = (C / (6 * 20))^0.5 [from D* = 20*N* and C = 6*N*D*]

    optimal_n = (compute_flops / (6 * 20)) ** 0.5
    optimal_d = 20 * optimal_n

    return {
        'compute_flops':    compute_flops,
        'compute_pflops':   compute_flops / 1e15,
        'optimal_params':   optimal_n,
        'optimal_params_M': optimal_n / 1e6,
        'optimal_tokens':   optimal_d,
        'optimal_tokens_B': optimal_d / 1e9,
    }


def flops_per_step(n_params: int, batch_tokens: int) -> float:
    """
    Approximate FLOPs for one training step.
    Rule of thumb: 6 * N * batch_tokens.
    (2 for forward, 4 for backward — backward is ~2× forward)
    """
    return 6 * n_params * batch_tokens


def plan_training_run(
    n_params:        int,
    gpu_flops_per_s: float,
    gpu_count:       int,
    hours:           float,
    batch_tokens:    int,
) -> dict:
    """
    Given hardware and time budget, compute:
    - Total FLOPs available
    - Number of training steps
    - Total tokens processed
    - How close to Chinchilla-optimal we get
    """
    total_flops    = gpu_flops_per_s * gpu_count * hours * 3600
    flops_per_stp  = flops_per_step(n_params, batch_tokens)
    total_steps    = int(total_flops / flops_per_stp)
    total_tokens   = total_steps * batch_tokens
    chinchilla     = chinchilla_optimal(total_flops)

    token_ratio    = total_tokens / chinchilla['optimal_tokens']

    print(f"\nTraining Run Plan")
    print(f"{'─'*50}")
    print(f"  Model parameters:    {n_params/1e6:.1f}M")
    print(f"  Hardware:            {gpu_count}× GPU @ {gpu_flops_per_s/1e12:.0f} TFLOP/s")
    print(f"  Time budget:         {hours:.1f} hours")
    print(f"  Total FLOPs:         {total_flops/1e18:.2f} EFLOPs")
    print(f"  Total steps:         {total_steps:,}")
    print(f"  Total tokens:        {total_tokens/1e6:.0f}M")
    print(f"{'─'*50}")
    print(f"  Chinchilla-optimal N: {chinchilla['optimal_params_M']:.1f}M params")
    print(f"  Chinchilla-optimal D: {chinchilla['optimal_tokens_B']:.2f}B tokens")
    print(f"  Our token ratio:      {token_ratio:.2f}× optimal")
    if token_ratio < 0.1:
        print(f"  ⚠ Severely undertrained — consider smaller model")
    elif token_ratio < 0.5:
        print(f"  ⚠ Undertrained — model could learn more with more data")
    elif token_ratio > 2.0:
        print(f"  ⚠ Overtrained on this dataset — model may be memorizing")
    else:
        print(f"  ✓ Near compute-optimal")

    return {
        'total_steps':  total_steps,
        'total_tokens': total_tokens,
        'token_ratio':  token_ratio,
    }


# Our nano model on a consumer GPU
plan = plan_training_run(
    n_params=10_700_000,    # 10.7M nano GPT
    gpu_flops_per_s=20e12,  # RTX 3090: ~20 TFLOP/s (BF16)
    gpu_count=1,
    hours=1.0,              # 1 hour of training
    batch_tokens=8 * 256,   # batch_size=8, seq_len=256 → 2048 tokens/step
)

# Output (approximately):
# Training Run Plan
# ──────────────────────────────────────────────────────
#   Model parameters:    10.7M
#   Hardware:            1× GPU @ 20 TFLOP/s
#   Time budget:         1.0 hours
#   Total FLOPs:         0.14 EFLOPs
#   Total steps:         6,738
#   Total tokens:        13.8M
# ──────────────────────────────────────────────────────
#   Chinchilla-optimal N: 1.1M params
#   Chinchilla-optimal D: 22.1M tokens
#   Our token ratio:      0.63× optimal
#   ⚠ Undertrained — model could learn more with more data

### The implication for the nano model

[[Our 10.7M parameter nano model is Chinchilla-optimal at $D^* = 20 \times 10.7M
= 214M$ tokens.]{.mark} TinyShakespeare has ~1M tokens — about 0.5% of what we need.
This means:

1. The model will overfit TinyShakespeare after a few epochs
2. For a real compute-optimal run, we need a much larger corpus
3. The training runs in this series are for learning purposes — they will
   demonstrate the mechanics but not produce a model that generalizes well

For the capstone (Tutorial 17), we will use a larger dataset and
Chinchilla-optimal training duration.

---

## 4. Gradient Accumulation

Sometimes you want a larger effective batch size than fits in GPU memory.
Gradient accumulation runs $k$ forward/backward passes before each optimizer
step, accumulating gradients — equivalent to a batch $k$ times larger:

$$\text{effective\_batch\_size} = \text{batch\_size} \times k$$

The key implementation detail: do not call `optimizer.zero_grad()` or
`optimizer.step()` until all $k$ accumulation steps are done. And divide
the loss by $k$ before backward — otherwise the gradient scale grows
with $k$, making the effective LR $k$ times larger:

In [ ]:
accumulation_steps = 4   # effective batch = 4× batch_size
optimizer.zero_grad()

for micro_step in range(accumulation_steps):
    x, y   = get_batch()
    _, loss = model(x, y)

    # Divide loss by accumulation steps — keeps gradient scale constant
    loss = loss / accumulation_steps
    loss.backward()   # gradients accumulate (not zeroed between micro-steps)

# After all micro-steps
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
optimizer.zero_grad()

[[**The `zero_grad` placement bug:**]{.underline} If you call `optimizer.zero_grad()` inside
the accumulation loop (before each micro-step), gradients are zeroed between
micro-steps and you only get gradients from the last micro-step — not the sum.
This is the exact bug mentioned in the README. The loss appears to train
normally (you never see an error) but you are not actually accumulating.

**Verifying accumulation is correct:**

In [ ]:
# Method: compare loss with accumulation to loss with a true large batch

# True large batch (batch_size=32)
x_large, y_large = get_batch(batch_size=32)
model.zero_grad()
_, loss_true = model(x_large, y_large)
loss_true.backward()
true_grad_norm = sum(p.grad.norm()**2 for p in model.parameters()
                     if p.grad is not None).sqrt().item()

# Accumulated (4 micro-batches of 8)
model.zero_grad()
for i in range(4):
    x_micro = x_large[i*8:(i+1)*8]
    y_micro = y_large[i*8:(i+1)*8]
    _, loss_micro = model(x_micro, y_micro)
    (loss_micro / 4).backward()
accum_grad_norm = sum(p.grad.norm()**2 for p in model.parameters()
                      if p.grad is not None).sqrt().item()

print(f"True large-batch grad norm:  {true_grad_norm:.6f}")
print(f"Accumulated grad norm:        {accum_grad_norm:.6f}")
# Should be approximately equal (small differences due to BN, if any)

---

## 5. The Complete Pretraining Loop

Everything assembled:

In [ ]:
import torch
import torch.nn as nn
import math
import time
import numpy as np
from pathlib import Path
from dataclasses import dataclass

from tutorial_02 import GPT, NanoGPTConfig
from tutorial_07 import PretrainingDataset, make_dataloader
from training_logger import TrainingLogger


@dataclass
class TrainingConfig:
    # Model
    model_config: NanoGPTConfig = None

    # Optimization
    max_lr:           float = 3e-4
    min_lr:           float = 3e-5
    weight_decay:     float = 0.1
    beta1:            float = 0.9
    beta2:            float = 0.95
    grad_clip:        float = 1.0

    # Schedule
    warmup_steps:     int   = 100
    max_steps:        int   = 5000

    # Batch
    batch_size:       int   = 8
    accumulation:     int   = 1     # gradient accumulation steps

    # Data
    block_size:       int   = 256
    num_workers:      int   = 2

    # Logging
    eval_every:       int   = 500
    eval_batches:     int   = 20
    log_every:        int   = 10
    checkpoint_every: int   = 1000

    # Paths
    data_dir:   str = 'data/shakespeare'
    run_dir:    str = 'runs/nano_gpt'

    def __post_init__(self):
        if self.model_config is None:
            self.model_config = NanoGPTConfig()


def compute_grad_stats(model):
    total_sq, ratios = 0.0, []
    for mod in model.modules():
        if isinstance(mod, nn.Linear) and mod.weight.grad is not None:
            g = mod.weight.grad.norm().item()
            w = mod.weight.norm().item()
            total_sq += g ** 2
            ratios.append(g / (w + 1e-8))
    gnorm = total_sq ** 0.5
    return gnorm, float(np.mean(ratios)) if ratios else 0.0, \
           float(np.min(ratios)) if ratios else 0.0, \
           float(np.max(ratios)) if ratios else 0.0


def pretrain(cfg: TrainingConfig):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    print(f"Device: {device}  |  dtype: {dtype}")

    # ---- Model ----
    model = GPT(cfg.model_config).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_params/1e6:.1f}M parameters")

    # ---- Optimizer ----
    # Separate weight decay from bias/layernorm parameters
    # Biases and LN parameters should NOT be weight-decayed
    decay_params = [p for n, p in model.named_parameters()
                    if p.dim() >= 2]   # weight matrices
    nodecay_params = [p for n, p in model.named_parameters()
                      if p.dim() < 2]  # biases, LN scales

    optimizer = torch.optim.AdamW([
        {'params': decay_params,   'weight_decay': cfg.weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0},
    ], lr=cfg.max_lr, betas=(cfg.beta1, cfg.beta2))

    print(f"Optimizer: {len(decay_params)} decay params, "
          f"{len(nodecay_params)} no-decay params")

    # ---- Schedule ----
    scheduler = make_cosine_schedule(
        optimizer, cfg.max_lr, cfg.min_lr,
        cfg.warmup_steps, cfg.max_steps
    )

    # ---- Data ----
    from tutorial_03 import Tokenizer
    tok = Tokenizer.load('nano_tokenizer.json')

    train_ds = PretrainingDataset.from_jsonl(
        cfg.data_dir, tok, cfg.block_size, split='train', buffer_size=500
    )
    val_ds = PretrainingDataset.from_jsonl(
        cfg.data_dir, tok, cfg.block_size, split='val', buffer_size=100
    )
    train_loader = make_dataloader(train_ds, cfg.batch_size, cfg.num_workers)
    val_loader   = make_dataloader(val_ds,   cfg.batch_size, 1)

    # ---- Logging ----
    logger = TrainingLogger(
        run_dir=cfg.run_dir,
        run_name='nano_gpt_pretrain',
        dashboard_url='http://localhost:8000',
    )

    # ---- Scaling law plan ----
    batch_tokens = cfg.batch_size * cfg.block_size * cfg.accumulation
    plan_training_run(n_params, 20e12, 1,
                      cfg.max_steps * batch_tokens / (20e12 * 3600),
                      batch_tokens)

    # ---- Training loop ----
    Path(cfg.run_dir).mkdir(parents=True, exist_ok=True)
    model.train()
    total_tokens = 0
    step         = 0
    best_eval    = float('inf')
    train_iter   = iter(train_loader)

    while step < cfg.max_steps:
        t0 = time.time()

        # ---- Gradient accumulation ----
        optimizer.zero_grad()
        step_loss = 0.0

        for micro_step in range(cfg.accumulation):
            try:
                x, y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)   # new epoch
                x, y = next(train_iter)

            x, y = x.to(device), y.to(device)

            with torch.autocast(device_type=device.type, dtype=dtype):
                _, loss = model(x, y)

            loss = loss / cfg.accumulation
            loss.backward()
            step_loss    += loss.item()
            total_tokens += x.numel()

        # ---- Gradient stats (before clip) ----
        gnorm, mean_r, min_r, max_r = compute_grad_stats(model)

        # ---- Clip and step ----
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()

        # ---- Timing ----
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.time() - t0
        tps     = total_tokens / (step + 1) / elapsed * (step + 1)  # approx

        lr = optimizer.param_groups[0]['lr']

        # ---- Log ----
        if step % cfg.log_every == 0:
            logger.log_step(
                step=step,
                train_loss=step_loss,
                learning_rate=lr,
                global_grad_norm=gnorm,
                tokens_per_sec=x.numel() / elapsed,
                mean_grad_ratio=mean_r,
                min_grad_ratio=min_r,
                max_grad_ratio=max_r,
                gpu_memory_gb=torch.cuda.memory_allocated()/1e9
                              if device.type == 'cuda' else 0.0,
            )

        # ---- Eval ----
        if step % cfg.eval_every == 0:
            model.eval()
            eval_losses = []
            with torch.no_grad():
                for i, (xv, yv) in enumerate(val_loader):
                    if i >= cfg.eval_batches:
                        break
                    with torch.autocast(device_type=device.type, dtype=dtype):
                        _, vl = model(xv.to(device), yv.to(device))
                    eval_losses.append(vl.item())
            eval_loss = float(np.mean(eval_losses))
            model.train()

            logger.log_step(
                step=step, train_loss=step_loss,
                learning_rate=lr, global_grad_norm=gnorm,
                tokens_per_sec=x.numel() / elapsed,
                eval_loss=eval_loss,
            )

            print(
                f"step {step:5d}  "
                f"train={step_loss:.4f}  eval={eval_loss:.4f}  "
                f"lr={lr:.2e}  gnorm={gnorm:.3f}  "
                f"ρ=[{min_r:.1e},{max_r:.1e}]"
            )

            # Save best checkpoint
            if eval_loss < best_eval:
                best_eval = eval_loss
                torch.save({
                    'step':       step,
                    'model':      model.state_dict(),
                    'optimizer':  optimizer.state_dict(),
                    'scheduler':  scheduler.state_dict(),
                    'config':     cfg,
                    'eval_loss':  eval_loss,
                }, f'{cfg.run_dir}/best_checkpoint.pt')

        # ---- Periodic checkpoint ----
        if step % cfg.checkpoint_every == 0 and step > 0:
            torch.save({
                'step':      step,
                'model':     model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
            }, f'{cfg.run_dir}/checkpoint_step{step:05d}.pt')

        step += 1

    # ---- Final ----
    logger.close()
    print(f"\nPretraining complete.")
    print(f"  Total tokens: {total_tokens:,}")
    print(f"  Best eval loss: {best_eval:.4f}")
    print(f"  Checkpoint: {cfg.run_dir}/best_checkpoint.pt")

    return model


if __name__ == '__main__':
    cfg   = TrainingConfig()
    model = pretrain(cfg)

### Checkpoint format

The checkpoint dictionary contains everything needed to resume training:

In [ ]:
def load_checkpoint(path: str, model, optimizer, scheduler):
    """Resume training from a checkpoint."""
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    step = ckpt['step']
    print(f"Resumed from step {step}  eval_loss={ckpt.get('eval_loss', '?')}")
    return step

[[**Always save the optimizer state.**]{.mark} The Adam moments ($m_t$, $v_t$) encode
[the recent gradient history for every parameter.]{.underline} Restarting without them
means the first 100–200 steps after a reload behave like fresh training —
the optimizer takes large, unstable steps until the moments rebuild. For
a long training run, this lost context is expensive.

---

## Summary

| Concept | Key detail |
|---|---|
| FP16 overflow | Max value 65504 — activations can exceed this. Requires `GradScaler`. |
| BF16 dynamic range | Same 8-bit exponent as FP32 — never overflows. No `GradScaler` needed. |
| `torch.autocast` | Casts matmul/conv to BF16; sensitive ops (softmax, LN) stay FP32. |
| `GradScaler` | For FP16 only: scales loss up before backward, down before optimizer step. |
| Adam warmup reason | Bias correction at $t=1$ amplifies first gradient 10×. Warmup keeps early steps small. |
| Warmup duration | 1–2% of total steps. Too long delays learning; too short causes early instability. |
| Cosine decay formula | $\text{min\_lr} + \frac{1}{2}(\text{max\_lr} - \text{min\_lr})(1 + \cos(\pi t / T))$ |
| `min_lr` choice | 10% of `max_lr` — Chinchilla convention. Keeps updates meaningful to the end. |
| Chinchilla $D^* = 20N$ | Compute-optimal training: 20 tokens per parameter. |
| FLOPs per token | $\approx 6N$ per training token (2× forward, 4× backward). |
| Gradient accumulation | Divide loss by $k$ before backward. `zero_grad()` outside the loop. |
| Decay vs no-decay params | Weight matrices: weight decay. Biases + LN scales: no weight decay. |
| Checkpoint contents | Model + optimizer + scheduler state. Load all three to resume correctly. |

---

## Exercises

**1.** Run the pretraining loop with `dtype=torch.float32` and
`dtype=torch.bfloat16` and compare throughput (tokens/sec). On a CUDA
GPU, BF16 should be significantly faster due to tensor core utilization.
Report the speedup ratio.

**2.** Plot the LR schedule for four configurations:
`warmup_steps` ∈ {10, 100, 500} and `min_lr` ∈ {0, 0.1×max, 0.5×max}.
For each, run 500 training steps and plot the loss curve. Confirm that
too little warmup causes instability in early steps and `min_lr=0`
causes the model to plateau earlier than `min_lr=0.1×max`.

**3.** Verify the gradient accumulation equivalence: train for 100 steps with
`batch_size=32, accumulation=1` and separately with `batch_size=8,
accumulation=4`. Both have effective batch size 32. Compare the final
loss and gradient norm trace. They should be nearly identical.

**4.** Use `plan_training_run` to compute the Chinchilla-optimal training
duration for the nano model on your machine. Then run for exactly that
many steps and compare the final eval loss to a run that uses 5× more
steps (overtrained). Confirm that overtraining on TinyShakespeare causes
the eval/train gap to widen.

**5.** Implement learning rate **restart** (SGDR — Stochastic Gradient Descent
with Warm Restarts): after each full cosine cycle, reset the LR to
`max_lr` and start a new cosine decay with period $2\times$ the
previous period. Compare loss curves between standard cosine and SGDR
on 5000 training steps.

**6.** Add a `detect_stale_checkpoint` function that loads a checkpoint and
checks whether the optimizer state is consistent with the model state:
compare the number of parameters in `model.state_dict()` vs the number
of parameter groups in `optimizer.state_dict()`. If they differ (which
happens when you change the model architecture between runs), print a
warning and re-initialize the optimizer rather than loading the stale state.